# 📓 Notebook A1 — LLM Providers Guide

> **Module:** AI Engineering · **Type:** Appendix · **Estimated time:** 30–45 min · **Difficulty:** Intermediate

By default every AI notebook in this course uses an offline `MockLLM` so you can run the code without internet, an API key, or a credit card. That's perfect for *learning the patterns*. The moment you want **real intelligence** in the answers, you swap one line — and this notebook is the reference for that swap.

You'll learn how to use:

- 🟢 **OpenAI** — the most popular hosted API; great default for most work.
- 🟠 **Anthropic** (Claude) — strong on reasoning, long context, careful tone.
- 🔵 **Google** (Gemini) — competitive on cost, especially for high-volume tasks.
- 🟣 **Ollama** — run *local* open-source LLMs on your own machine. No internet, no per-call cost.

All four implementations live in [`llm_providers.py`](../llm_providers.py) at the root of the refined course and share the **same `chat()` interface**. Swapping providers in any of NB 18 / 19 / 20 / 21 / 22 / 27 is a *one-line change*.

## 🎯 Learning objectives

By the end of this notebook you can:

1. Pick the right provider for cost, latency, quality, and data-privacy constraints.
2. Install and authenticate each of the four providers safely.
3. Use the unified `llm_providers` module to swap providers without changing application code.
4. Estimate **cost** for a known workload across providers.
5. Run a fully **local** open-source LLM via Ollama.
6. Use the same pattern for **embeddings** (NB 19) — hosted vs local.

## ✅ Prerequisites

NB 18 (AI workflows). Helpful: NB 19 (embeddings), NB 22 (evaluation).

## 1. The unified `chat()` interface

Every provider in this course implements one minimal method:

```python
response = llm.chat(
    messages=[
        {"role": "system", "content": "You are helpful."},
        {"role": "user",   "content": "Hello!"},
    ],
    temperature=0.0,
    max_tokens=512,
)

response["text"]        # the assistant's reply (str)
response["model"]       # actual model name returned
response["tokens_in"]   # input tokens billed (or estimated)
response["tokens_out"]  # output tokens billed
response["latency_s"]   # wall-clock time, seconds
```

That contract is the same for all five classes — `MockLLM`, `OpenAILLM`, `AnthropicLLM`, `GoogleLLM`, `OllamaLLM`. Your notebook code does not change when you swap them; only the *constructor call* changes.

## 2. Smoke test — the `MockLLM` works offline

Let's first verify the module imports and the offline mock works. Everything else in this notebook is *reference code* that you only run if you have the corresponding key/SDK.

In [ ]:
import sys
from pathlib import Path

# Make the refined_course/ root importable
sys.path.insert(0, str(Path("..").resolve()))

from llm_providers import MockLLM, get_llm

llm = MockLLM()
r = llm.chat(messages=[
    {"role": "system", "content": "Return JSON with keys sentiment and topic."},
    {"role": "user",   "content": "My invoice is wrong, please refund."},
])
print(f"text       : {r['text']}")
print(f"model      : {r['model']}")
print(f"tokens_in  : {r['tokens_in']}")
print(f"tokens_out : {r['tokens_out']}")
print(f"latency_s  : {r['latency_s']:.4f}")


## 3. Picking a provider — the decision table

| Criterion | OpenAI | Anthropic | Google | Ollama (local) |
|---|---|---|---|---|
| **Latency** (small models) | ~0.5–2 s | ~0.5–2 s | ~0.5–1.5 s | depends on hardware |
| **Per-token cost** | low–medium | low–medium | very low (Flash) | $0 |
| **Quality (small models)** | gpt-4o-mini ★★★★ | claude-haiku ★★★★ | gemini-flash ★★★ | Llama 3.2 3B ★★ |
| **Quality (flagship)** | gpt-4o ★★★★★ | claude-sonnet ★★★★★ | gemini-pro ★★★★ | Llama 3.1 70B ★★★★ |
| **Long-context** | 128K | 200K | 1M+ | depends on model |
| **Data privacy** | sent to API | sent to API | sent to API | **stays local** |
| **Internet required** | yes | yes | yes | **no** |
| **API key required** | yes | yes | yes | **no** |
| **Best for** | default; reliable, well-documented | reasoning-heavy; longer drafts; careful tone | high-volume / cost-sensitive | privacy-sensitive; offline; tinkering |

### When to pick which

- **Default? OpenAI.** Largest ecosystem, best documentation, mature SDK.
- **Need to process *long* documents?** Anthropic (200K context) or Gemini (1M).
- **Cost-sensitive at high volume?** Gemini Flash is often cheapest.
- **Data must not leave your machine?** Ollama. Period.
- **Demoing without any keys?** Ollama (or stay on `MockLLM`).


## 4. 🟢 OpenAI — the default

```bash
pip install openai
export OPENAI_API_KEY=sk-...        # in your shell, BEFORE launching Jupyter
```

> ⚠️ **Never** paste an API key into a notebook cell that you might commit to git. Use environment variables (or a `.env` file with `python-dotenv`).

In [ ]:
# Reference code — uncomment after `pip install openai` and setting the key.
# from llm_providers import OpenAILLM
#
# llm = OpenAILLM(model="gpt-4o-mini")
# r = llm.chat(messages=[
#     {"role": "system", "content": "You are a concise assistant."},
#     {"role": "user",   "content": "Why are LLMs good at translation?"},
# ])
# print(r["text"])
# print(f"\nTokens: in={r['tokens_in']}, out={r['tokens_out']}  |  cost ≈ ${r['tokens_in']*0.0006/1000 + r['tokens_out']*0.0024/1000:.5f}")

print("(cell intentionally inactive — uncomment after install + auth)")


### Common OpenAI models

| Model | Speed | Cost | Notes |
|---|---|---|---|
| `gpt-4o-mini` | very fast | cheap | Default choice. Excellent quality-per-dollar. |
| `gpt-4o` | fast | medium | Smartest of the chat models; long context. |
| `o1-mini`, `o3-mini` | slower (reasoning) | medium | Use for math / multi-step logic. |
| `text-embedding-3-small` | very fast | very cheap | The embeddings model from NB 19. |

> 💡 **Picking the smallest model that works** is the single biggest cost lever. `gpt-4o-mini` handles 80% of classification / summarisation tasks fine.

## 5. 🟠 Anthropic (Claude)

```bash
pip install anthropic
export ANTHROPIC_API_KEY=sk-ant-...
```

Anthropic's API splits the system prompt into its own `system=` parameter. The wrapper in `llm_providers.py` does that translation for you — your code stays in the OpenAI-style format.

In [ ]:
# Reference code — same shape as OpenAILLM.
# from llm_providers import AnthropicLLM
#
# llm = AnthropicLLM(model="claude-haiku-4-5-20251001")
# r = llm.chat(messages=[
#     {"role": "system", "content": "You are a careful copy editor."},
#     {"role": "user",   "content": "Improve this: 'we will done that quickly'."},
# ])
# print(r["text"])

print("(cell intentionally inactive — uncomment after install + auth)")


### Common Anthropic models

| Model | Speed | Cost | Notes |
|---|---|---|---|
| `claude-haiku-4-5-20251001` | very fast | cheap | Great default for classification + light reasoning. |
| `claude-sonnet-4-6` | fast | medium | Best balance for production. |
| `claude-opus-4-6` | slower | high | Reasoning-heavy tasks, longer drafts. |

Claude's strengths: **long context** (200K tokens by default), careful tone in long-form writing, well-behaved JSON output.

## 6. 🔵 Google (Gemini)

```bash
pip install google-generativeai
export GOOGLE_API_KEY=...     # or GEMINI_API_KEY — both supported by the wrapper
```

The wrapper translates OpenAI-style messages into Gemini's `parts` format, and surfaces system instructions via the `system_instruction` parameter.

In [ ]:
# Reference code — same call shape as OpenAILLM/AnthropicLLM.
# from llm_providers import GoogleLLM
#
# llm = GoogleLLM(model="gemini-2.0-flash")
# r = llm.chat(messages=[
#     {"role": "system", "content": "Return one JSON object with keys sentiment and topic."},
#     {"role": "user",   "content": "Add support for Markdown please."},
# ])
# print(r["text"])

print("(cell intentionally inactive — uncomment after install + auth)")


### Common Gemini models

| Model | Speed | Cost | Notes |
|---|---|---|---|
| `gemini-2.0-flash` | very fast | very cheap | Default; great for high-volume classification. |
| `gemini-2.0-flash-lite` | very fast | cheapest | Lighter — fine for short prompts. |
| `gemini-2.5-pro` | slower | medium | Smartest Gemini; **1M+ token context**. |

Gemini's superpower: that **1M-token context window**. Whole codebases or hour-long meeting transcripts fit in one prompt.

## 7. 🟣 Ollama — fully local LLMs

Ollama gives you the *same chat-style API* against open-source models running entirely on your own machine. **No internet. No API key. No per-call cost.** It is the right answer when:

- The data is sensitive and must not leave the box.
- You want to demo without any signup.
- You're experimenting with prompts and want zero variable cost.

### One-time setup

```bash
# 1. Install the Ollama server
#    https://ollama.com — installer for macOS / Linux / Windows.

# 2. Pull a model (the first call is slow; subsequent are local)
ollama pull llama3.2:3b              # 2 GB, runs on CPU
ollama pull qwen2.5:7b               # 4 GB, better quality
ollama pull deepseek-r1:8b           # reasoning-heavy

# 3. (Ollama serves on http://localhost:11434 by default — no action needed.)

# 4. Install the Python client
pip install ollama
```

In [ ]:
# Reference code — uncomment after Ollama is running and a model is pulled.
# from llm_providers import OllamaLLM
#
# llm = OllamaLLM(model="llama3.2:3b")
# r = llm.chat(messages=[
#     {"role": "system", "content": "Reply in JSON with key 'sentiment' only."},
#     {"role": "user",   "content": "Loving the new dashboard."},
# ])
# print(r["text"])
# print(f"\nLatency: {r['latency_s']:.2f}s  (CPU-only is ~1-5s; GPU brings it under 1s)")

print("(cell intentionally inactive — uncomment after Ollama is running)")


### Which local model should I pick?

| Model | Disk | RAM/VRAM | Quality (rough) | Good for |
|---|---|---|---|---|
| `llama3.2:3b` | 2 GB | 4 GB | ★★ | Demos, classification, short replies |
| `qwen2.5:7b` | 4 GB | 8 GB | ★★★ | Most light-AI work; good multilingual |
| `llama3.1:8b` | 5 GB | 8 GB | ★★★ | Stronger reasoning |
| `gemma2:9b` | 5 GB | 10 GB | ★★★ | Strong instruction-following |
| `qwen2.5:32b` | 20 GB | 32 GB | ★★★★ | Production-quality (needs a GPU) |

> 💡 **Start with the 3B model.** It runs on a laptop CPU and is good enough for prompt-engineering experiments. Move up only if quality is the bottleneck.

## 8. Comparing two providers on the same task

Once your application uses the unified interface, **A/B-testing models becomes trivial** — same loop, two different clients. The cell below shows the pattern (using `MockLLM` twice; in real life you'd use two different providers).

In [ ]:
# A/B template: same prompts, two providers, compare outputs.
from llm_providers import MockLLM

provider_a = MockLLM(seed=0)
provider_b = MockLLM(seed=42)              # in real life: OpenAILLM() and AnthropicLLM()

queries = [
    "Refund please.",
    "Why is loading so slow?",
    "Could you add CSV export?",
]

rows = []
for q in queries:
    a = provider_a.chat(messages=[
        {"role": "system", "content": "Return JSON with sentiment and topic."},
        {"role": "user",   "content": q},
    ])
    b = provider_b.chat(messages=[
        {"role": "system", "content": "Return JSON with sentiment and topic."},
        {"role": "user",   "content": q},
    ])
    rows.append((q, a["text"], b["text"]))

for q, ra, rb in rows:
    print(f"Q: {q}")
    print(f"  A → {ra}")
    print(f"  B → {rb}")
    print()


**Why this matters.** The same `messages` list works against any provider; the same output dict comes back. You can keep your prompts and evaluation harness *exactly* as in NB 22 and swap models for a head-to-head comparison.

> 🎯 **Evaluation-driven model selection.** Run your golden set (NB 22) against each provider, compare accuracy / latency / cost. The "best" model is the cheapest one that passes the eval — not necessarily the most expensive one.

## 9. Cost estimation — back-of-envelope per provider

A simple model: cost = `(tokens_in / 1000) * price_in + (tokens_out / 1000) * price_out`. Approximate prices (USD per 1K tokens, check the provider for current values — they change):

In [ ]:
# Approximate per-1K-token prices in USD (rough Nov-2024 figures — verify with the provider!)
PRICES = {
    "OpenAI gpt-4o-mini":          {"in": 0.000150, "out": 0.000600},
    "OpenAI gpt-4o":               {"in": 0.002500, "out": 0.010000},
    "Anthropic claude-haiku":      {"in": 0.000800, "out": 0.004000},
    "Anthropic claude-sonnet":     {"in": 0.003000, "out": 0.015000},
    "Google gemini-2.0-flash":     {"in": 0.000075, "out": 0.000300},
    "Google gemini-2.5-pro":       {"in": 0.001250, "out": 0.005000},
    "Ollama llama3.2:3b (local)":  {"in": 0.000000, "out": 0.000000},
}

# A realistic monthly workload: 50,000 messages × 500 tokens in, 100 tokens out
TOKENS_IN_PER_MSG  = 500
TOKENS_OUT_PER_MSG = 100
N_MSGS_PER_MONTH   = 50_000

print(f"Cost per provider for {N_MSGS_PER_MONTH:,} messages/month\n"
      f"  ({TOKENS_IN_PER_MSG} in, {TOKENS_OUT_PER_MSG} out per message):\n")
print(f"  {'Provider':<32}{'monthly $':>12}")
print(f"  {'-'*32}{'-'*12}")
for name, p in PRICES.items():
    total = N_MSGS_PER_MONTH * (TOKENS_IN_PER_MSG/1000 * p["in"]
                                + TOKENS_OUT_PER_MSG/1000 * p["out"])
    print(f"  {name:<32}${total:>10,.2f}")


**The takeaways.** At ~50K messages/month with these per-message sizes:

- **Gemini Flash** is the cheapest hosted option (~$5/mo).
- **gpt-4o-mini** is competitive on cost and very reliable.
- **Flagship models** (gpt-4o, claude-sonnet) are 10–30× more expensive — pay for them only when the small models fail your eval.
- **Local (Ollama)** is free per call but ties up your hardware. The break-even depends on your usage; at ~100K calls/month, a $400 GPU pays for itself in under a year.

## 10. The embedding side — same idea for NB 19

Embeddings (the vector representations used in retrieval) have the same hosted-vs-local choice.

In [ ]:
from llm_providers import MockEmbedder, get_embedder

emb = MockEmbedder(dim=128)
vecs = emb.embed([
    "How do I cancel my subscription?",
    "How can I end my plan?",
    "How is the weather today?",
])
print(f"shape: {vecs.shape}")
# Cosine similarity (L2-normalised → dot product)
import numpy as np
print(f"sim('cancel sub', 'end plan') = {float(vecs[0] @ vecs[1]):.3f}")
print(f"sim('cancel sub', 'weather')  = {float(vecs[0] @ vecs[2]):.3f}")


### Real embedders

| Embedder | Model | Cost | Where it runs |
|---|---|---|---|
| `OpenAIEmbedder` | `text-embedding-3-small` | $0.00002 / 1K tokens (very cheap) | OpenAI cloud |
| `OpenAIEmbedder` | `text-embedding-3-large` | $0.00013 / 1K tokens | OpenAI cloud |
| `LocalEmbedder` | `all-MiniLM-L6-v2` (80 MB) | free per call | your machine |
| `LocalEmbedder` | `BAAI/bge-large-en-v1.5` (1.3 GB) | free per call | your machine |

```python
from llm_providers import OpenAIEmbedder, LocalEmbedder

# Hosted:
emb = OpenAIEmbedder("text-embedding-3-small")

# Local (one-time download on first call):
# pip install sentence-transformers
emb = LocalEmbedder("all-MiniLM-L6-v2")

vectors = emb.embed(["my first doc", "my second doc"])
```

> 💡 **For most semantic-search tasks, `all-MiniLM-L6-v2` is good enough.** It's 22M parameters, runs on a CPU, and matches the quality of much larger models for short-to-medium texts.

## 11. Swapping into the existing AI notebooks

Every existing AI notebook (NB 18, 19, 20, 21, 22, 27) defines an `llm = MockLLM()` near the top. Replace exactly that one line:

```python
# Before:
from llm_providers import MockLLM
llm = MockLLM()

# After (any one of these):
from llm_providers import OpenAILLM
llm = OpenAILLM(model="gpt-4o-mini")

from llm_providers import AnthropicLLM
llm = AnthropicLLM(model="claude-haiku-4-5-20251001")

from llm_providers import GoogleLLM
llm = GoogleLLM(model="gemini-2.0-flash")

from llm_providers import OllamaLLM
llm = OllamaLLM(model="llama3.2:3b")
```

The rest of the notebook is unchanged. That's the whole point of the unified interface.

> ⚠️ **Heads-up:** real LLMs are non-deterministic by default. Cell outputs in the existing notebooks (which were captured against `MockLLM`) will *not* match exactly when you swap to a real provider. That is *correct* — and exactly what the eval harness in NB 22 is for.

## 12. Production patterns you'll want eventually

A few patterns that pay back in real deployments:

| Pattern | What it does | Where to add it |
|---|---|---|
| **Caching** by `(prompt_hash, model)` | Skip the API call if you've seen the exact prompt before | A `functools.lru_cache` around `chat()` |
| **Retry with exponential backoff** | Survive transient 429 / 5xx errors | NB 7 has the reusable function |
| **Cost ceiling per request** | Refuse calls that would exceed a hard cap | Wrap the constructor in a factory |
| **Provider fallback chain** | Try Anthropic; on failure, fall back to OpenAI | A small loop over `[primary, fallback]` |
| **Streaming responses** | Show tokens as they generate (UX!) | Provider-specific; OpenAI / Anthropic both support it |

These are out of scope for the course's notebooks (which prioritise the *patterns*), but every one of them is 10–30 lines of code on top of what you have.

## 🧪 Exercises

### Exercise 1 — A provider-swap drill

Open NB 18 (`05_ai_engineering/18_ai_workflows.ipynb`). Find the cell that creates `llm = MockLLM()`. Change it to use `OpenAILLM(model="gpt-4o-mini")` *as a code comment only* — do not actually run it unless you have a key. Then write a 3-line cell beneath it that prints which provider is active.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
# Change the original line to:
# from llm_providers import OpenAILLM
# llm = OpenAILLM(model="gpt-4o-mini")

# Then add this confirmation cell:
print(f"Provider: {type(llm).__name__}")
print(f"Model   : {getattr(llm, 'model', 'mock-mini')}")
```

You'd see `Provider: OpenAILLM, Model: gpt-4o-mini` after the swap. Every other cell in the notebook continues to work without change — that's the value of the unified interface.
</details>

### Exercise 2 — Estimate cost for *your* expected workload

Pick a workload: say, **20,000 customer-feedback messages per month**, average **300 tokens in / 80 tokens out**. Estimate the monthly cost for each of the four hosted models in §9. Which is cheapest? Which would you actually ship and why?

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
TOKENS_IN, TOKENS_OUT, N = 300, 80, 20_000
for name, p in PRICES.items():
    cost = N * (TOKENS_IN/1000 * p["in"] + TOKENS_OUT/1000 * p["out"])
    print(f"{name:<32}${cost:.2f}")
```

You'll find Gemini Flash and `gpt-4o-mini` come out closest. Which one to ship depends on:

- **Quality on your eval set** (NB 22). The cheapest model that passes is the winner.
- **Vendor risk.** OpenAI has the longest track record but Google's SLA is excellent.
- **Existing infrastructure.** If your team is already on GCP, Gemini is a one-click integration.

This is exactly the kind of analysis to write up before any AI feature ships.
</details>

### Exercise 3 — A local-first development workflow

Imagine you're prototyping an AI feature on a flight (no internet). Write a function `local_first_chat(messages, prefer_local: bool = True)` that:

1. If `prefer_local` is True and `OllamaLLM` is available, use it.
2. Otherwise fall back to `OpenAILLM` if `OPENAI_API_KEY` is set.
3. Otherwise fall back to `MockLLM`.

Always return the same response dict.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
import os
from llm_providers import MockLLM, OpenAILLM, OllamaLLM

def local_first_chat(messages, *, prefer_local: bool = True, **kwargs):
    if prefer_local:
        try:
            return OllamaLLM(model="llama3.2:3b").chat(messages, **kwargs)
        except Exception:
            pass    # Ollama not installed / not running → fall through
    if os.getenv("OPENAI_API_KEY"):
        try:
            return OpenAILLM(model="gpt-4o-mini").chat(messages, **kwargs)
        except Exception:
            pass
    return MockLLM().chat(messages, **kwargs)


# Should pick the lowest-friction available option without crashing.
r = local_first_chat([{"role": "user", "content": "Hi!"}])
print(f"Used model: {r['model']}")
```

This is the pattern behind every "works offline, syncs when online" tool. The function tries options in order of *preference*; each attempt fails fast on a clean `ImportError` or `RuntimeError`, never throwing a traceback at the user.
</details>

## 🧠 Key takeaways

1. The course uses **`MockLLM` by default** so every notebook runs offline.
2. A single module (`llm_providers.py`) gives all five providers the **same `chat()` interface**.
3. **Swapping providers** is a **one-line change** in any of the AI notebooks.
4. **OpenAI** is the safest default; **Anthropic** for long context / careful drafts; **Gemini** for cost-sensitive high-volume; **Ollama** for offline / privacy.
5. **Always validate on your golden set** (NB 22) when comparing providers — cost without accuracy is meaningless.
6. **The smallest model that passes the eval wins.** Don't over-pay for capacity you don't use.
7. **Embeddings** have the same hosted-vs-local choice — pick `OpenAIEmbedder` or `LocalEmbedder` (sentence-transformers).
8. **Never commit API keys.** Use environment variables (`os.getenv`) or `.env` files.

## ✅ Self-assessment

- [ ] Pick the right provider for a given cost / privacy / latency requirement
- [ ] Install and authenticate any of OpenAI / Anthropic / Gemini
- [ ] Install Ollama and pull a local model
- [ ] Swap the LLM provider in any AI notebook with one line
- [ ] Estimate monthly cost for a given workload across providers
- [ ] Build a fallback chain (`local → hosted → mock`)

## 🚀 Where to go from here

You now have a portable LLM toolkit. The natural next steps:

1. **Set up a real key** for one hosted provider and re-run NB 22's evaluation harness against it. Compare with the MockLLM numbers.
2. **Try a local model** via Ollama on NB 18's inbox-triage task. Note the latency difference vs. the mock.
3. **Build a fallback** — your production code should try a fast provider, fall back to a robust one, never to a crash.
